## Q1(a) — Slope table

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from pathlib import Path
out = Path('.')
values = np.array([-1, -0.5, 0, 0.5, 1])
print('y vs x', *[f'{x:8g}' for x in values])
for y in values[::-1]:
    print(f'{y:4g}', *[f'{y-x*x+1:8.2f}' for x in values])

## Q1(b) — Computer slope field

In [ ]:
def f1(x,y):
    return y-x**2+1
xv=np.linspace(-2,2,25)
yv=np.linspace(-12,12,31)
X,Y=np.meshgrid(xv,yv)
S=f1(X,Y)
L=np.sqrt(1+S**2)
fig,ax=plt.subplots(figsize=(9,6))
ax.quiver(X,Y,1/L,S/L,angles='xy',pivot='mid',color='gray',alpha=.65)
ax.set(xlim=(-2,2),ylim=(-12,12),xlabel='x',ylabel='y',title='Q1(b): Direction field')
fig.tight_layout()
fig.savefig(out/'Q1b_direction_field.png')
plt.show()

## Q1(c) — Personalized solution curves

In [ ]:
d=5
def both(y0):
    a=solve_ivp(f1,(0,-2),[y0],t_eval=np.linspace(0,-2,400),rtol=1e-10,atol=1e-10)
    b=solve_ivp(f1,(0,2),[y0],t_eval=np.linspace(0,2,400),rtol=1e-10,atol=1e-10)
    return np.r_[a.t[::-1],b.t[1:]],np.r_[a.y[0,::-1],b.y[0,1:]]
fig,ax=plt.subplots(figsize=(9,6))
ax.quiver(X,Y,1/L,S/L,angles='xy',pivot='mid',color='gray',alpha=.4)
for y0 in sorted(set([-d,0,d])):
    xx,yy=both(y0)
    line,=ax.plot(xx,yy,lw=2,label=f'y(0)={y0}')
    ax.plot(0,y0,'o',color=line.get_color())
z=np.linspace(-2,2,300)
ax.plot(z,z*z-1,'k--',label='Zero slope: y=x²−1')
ax.set(xlim=(-2,2),ylim=(-12,12),xlabel='x',ylabel='y',title='Q1(c): Direction field and solution curves')
ax.legend()
fig.tight_layout()
fig.savefig(out/'Q1c_solutions.png')
plt.show()

## Q2(a) — Phase-plane vector field

In [ ]:
g=9.81
Lpend=1.0
def pend(t,state):
    th,om=state
    return [om,-g/Lpend*np.sin(th)]
thv=np.linspace(-np.pi,np.pi,25)
omv=np.linspace(-5,5,25)
TH,OM=np.meshgrid(thv,omv)
U=OM
V=-g/Lpend*np.sin(TH)
R=np.hypot(U,V)
R[R==0]=1
fig,ax=plt.subplots(figsize=(9,6))
ax.quiver(TH,OM,U/R,V/R,angles='xy',pivot='mid',color='gray')
ax.set(xlim=(-np.pi,np.pi),ylim=(-5,5),xlabel='θ (rad)',ylabel='ω (rad/s)',title='Q2(a): Pendulum phase-plane field')
fig.tight_layout()
fig.savefig(out/'Q2a_phase_field.png')
plt.show()

## Q2(b) — Numerical trajectories

In [ ]:
t=np.linspace(0,10,2001)
sols={a:solve_ivp(pend,(0,10),[a,0],t_eval=t,rtol=1e-11,atol=1e-11) for a in [0.1,1.0]}
fig,ax=plt.subplots(figsize=(9,5))
for a,s in sols.items():
    ax.plot(s.t,s.y[0],label=f'θ(0)={a}')
ax.set(xlabel='t (s)',ylabel='θ (rad)',title='Q2(b): Angular displacement')
ax.legend()
fig.tight_layout()
fig.savefig(out/'Q2b_theta_time.png')
plt.show()
fig,ax=plt.subplots(figsize=(9,6))
ax.quiver(TH,OM,U/R,V/R,angles='xy',pivot='mid',color='gray',alpha=.45)
for a,s in sols.items():
    ax.plot(s.y[0],s.y[1],lw=2,label=f'θ(0)={a}')
    ax.plot(a,0,'o')
ax.set(xlim=(-np.pi,np.pi),ylim=(-5,5),xlabel='θ (rad)',ylabel='ω (rad/s)',title='Q2(b): Phase trajectories')
ax.legend()
fig.tight_layout()
fig.savefig(out/'Q2b_phase_trajectories.png')
plt.show()

## Q2(c) — Linear comparison and error

In [ ]:
w0=np.sqrt(g/Lpend)
for a,s in sols.items():
    linear=a*np.cos(w0*t)
    err=np.abs(s.y[0]-linear)
    fig,(ax1,ax2)=plt.subplots(2,1,figsize=(9,7),sharex=True)
    ax1.plot(t,s.y[0],label='Nonlinear')
    ax1.plot(t,linear,'--',label='Linear')
    ax1.set(ylabel='θ (rad)',title=f'Q2(c): θ(0)={a}')
    ax1.legend()
    ax2.plot(t,err)
    ax2.set(xlabel='t (s)',ylabel='Absolute error (rad)')
    fig.tight_layout()
    fig.savefig(out/f'Q2c_comparison_{a}.png')
    plt.show()
    print(f'θ(0)={a}: maximum absolute error={err.max():.8f} rad')

## Q3(a) — First three Euler steps

In [ ]:
def f3(x,y):
    return -3*y+6*np.sin(x)
x=0.0
y=2.0
for n in range(3):
    slope=f3(x,y)
    ynext=y+0.5*slope
    print(f'{x:.1f}  {y:.8f}  {slope:.8f}  {ynext:.8f}')
    x+=0.5
    y=ynext

## Q3(b) — Euler method

In [ ]:
def euler(h):
    n=int(round(4/h))
    x=h*np.arange(n+1)
    y=np.zeros(n+1)
    y[0]=2
    for i in range(n):
        y[i+1]=y[i]+h*f3(x[i],y[i])
    return x,y
x,y=euler(0.5)
fig,ax=plt.subplots(figsize=(9,5))
ax.plot(x,y,'o-',label='Euler h=0.5')
ax.set(xlabel='x',ylabel='y',title='Q3(b): Euler approximation')
ax.legend()
fig.tight_layout()
fig.savefig(out/'Q3b_euler.png')
plt.show()

## Q3(c) — Built-in solver comparison

In [ ]:
reference=solve_ivp(f3,(0,4),[2],t_eval=np.linspace(0,4,1001),rtol=1e-11,atol=1e-11)
refnodes=solve_ivp(f3,(0,4),[2],t_eval=x,rtol=1e-11,atol=1e-11)
fig,ax=plt.subplots(figsize=(9,5))
ax.plot(reference.t,reference.y[0],label='solve_ivp')
ax.plot(x,y,'o--',label='Euler h=0.5')
ax.set(xlabel='x',ylabel='y',title='Q3(c): Euler versus built-in solver')
ax.legend()
fig.tight_layout()
fig.savefig(out/'Q3c_solver_comparison.png')
plt.show()
print('Maximum nodal error:',np.max(np.abs(y-refnodes.y[0])))

## Q3(d) — Step-size and stability comparison

In [ ]:
def exact(x):
    return 2.6*np.exp(-3*x)+1.8*np.sin(x)-0.6*np.cos(x)
fig,(ax1,ax2)=plt.subplots(2,1,figsize=(9,8),sharex=True)
xx=np.linspace(0,4,1001)
ax1.plot(xx,exact(xx),'k-',label='Exact')
for h in [0.5,0.25,0.1]:
    xh,yh=euler(h)
    err=np.abs(yh-exact(xh))
    ax1.plot(xh,yh,'o--',ms=3,label=f'h={h}')
    ax2.plot(xh,err,'o-',ms=3,label=f'h={h}')
    print(f'h={h}: max error={err.max():.8f}, final error={err[-1]:.8f}, factor={1-3*h:.2f}')
ax1.set(ylabel='y',title='Q3(d): Step-size comparison')
ax2.set(xlabel='x',ylabel='Absolute error',title='Absolute error versus exact solution')
ax1.legend()
ax2.legend()
fig.tight_layout()
fig.savefig(out/'Q3d_step_sizes.png')
plt.show()